# Milestone 2 lab, notebook 6 of 8: A normal day

## Where we are

| Notebook | What you do |
|---|---|
| 1. Empty in which rows | Write a check for empty cells that does not fire on good rows |
| 2. Unique on which columns | Find the receipt and the file that were counted twice |
| 3. A believable value | Catch 1,180 kg of bananas without catching 400 real bottles |
| 4. Rules of the business | Three rules a person must tell you. Each needs a join first |
| 5. Across two tables | Hold the receipts against the tills' own totals. Find the lines a join dropped |
| **6. A normal day** (you are here) | Find the store that sent nothing. Work out what a normal day is |
| 7. The false alarm | A holiday is not an error. See what silence costs |
| 8. The production version | The same checks as a tested program, and what is still wrong |

**You need:** the warehouse running. From the project folder: `docker compose up -d warehouse`.

**About ten minutes.**

In [1]:
from lab_helpers import bonus, money, prepare_week, sql, true_net_sales

prepare_week()

The bonus week is loaded: seven nights, Mon 24 to Sun 30 August. 24,384 sales lines.
The five standard checks ran on every night: 35 of 35 passed.


## Step 1: Look at the week from above

Every check so far looked at rows: is this row empty, twice, too big, against a rule? This notebook looks at something else: **how much data arrived**.

Here are the sales lines of the bonus week, one row per day, one column per store.

In [2]:
lines = sql("""
    SELECT sales_date, store_id, count(*) AS lines
    FROM clean.sales_lines
    WHERE sales_date BETWEEN DATE '2026-08-24' AND DATE '2026-08-30'
    GROUP BY sales_date, store_id
""")
lines.pivot(index="sales_date", columns="store_id", values="lines")

store_id,S01,S02,S03,S04,S05,S06
sales_date,,,,,,
2026-08-24,117.0,125.0,113.0,89.0,68.0,98.0
2026-08-25,132.0,119.0,95.0,NaN,61.0,84.0
2026-08-26,52.0,36.0,42.0,41.0,30.0,32.0
2026-08-27,NaN,110.0,121.0,124.0,84.0,108.0
2026-08-28,153.0,334.0,118.0,135.0,107.0,124.0
2026-08-29,170.0,179.0,142.0,172.0,108.0,141.0
2026-08-30,177.0,149.0,113.0,123.0,82.0,100.0


Four things in that table are strange. `NaN` means: no rows at all.

1. **Tuesday 25, `S04`:** no rows.
2. **Thursday 27, `S01`:** no rows.
3. **Wednesday 26:** every store is far lower than on any other day.
4. **Friday 28, `S02`:** 334 lines, far more than any other store-day.

All 35 standard checks were green that week.

Take the first one. **Ines Marchetti** manages `S04`. What did her store really sell that Tuesday?

In [3]:
missing = true_net_sales("S04", "2026-08-25")
print("S04 really sold", money(missing), "on Tuesday 25 August.")
print("The pipeline has no row for that day. Ines's bonus is", money(bonus(missing)), "short.")

S04 really sold $587.42 on Tuesday 25 August.
The pipeline has no row for that day. Ines's bonus is $5.87 short.


$587.42 of sales, and not one row. The store's back-office computer did not send the file.

Why did no check fail? Because **a check looks at rows, and there are no rows**. "The table is not empty" passed: five other stores sent theirs. A row that is not there cannot fail anything.

## Step 2: A check for rows that are not there

To find something missing, first write down what you **expect**: every store, on every day. Then take away what you have.

In [4]:
sql("""
    SELECT s.store_id, d.day AS sales_date, 'no sales lines' AS problem
    FROM clean.stores s
    CROSS JOIN (SELECT DISTINCT business_date AS day FROM clean.till_totals) d
    LEFT JOIN (SELECT DISTINCT store_id, sales_date FROM clean.sales_lines) l
      ON l.store_id = s.store_id AND l.sales_date = d.day
    WHERE l.store_id IS NULL
    ORDER BY d.day, s.store_id
""")

,store_id,sales_date,problem
0,S04,2026-08-25,no sales lines
1,S01,2026-08-27,no sales lines


Two rows: the two holes from the table above. This is the first check in the project that returns rows which do **not exist** in the data.

Keep both rows in mind. One of them is not a fault. That is the next notebook.

## Step 3: How many is too many?

Now the fourth strange thing. **Dana Okafor**'s store had 334 lines on Friday. Is that too many?

You cannot tell from the number alone. 334 would be absurd for a small store, and normal for a hypermarket. You need something to compare it with. Compare it with the same store on earlier Fridays.

In [5]:
fridays = sql("""
    SELECT sales_date, count(*) AS lines
    FROM clean.sales_lines
    WHERE store_id = 'S02' AND extract(isodow FROM sales_date) = 5
    GROUP BY sales_date
    ORDER BY sales_date
""")
fridays

,sales_date,lines
0,2026-07-31,159
1,2026-08-07,165
2,2026-08-14,141
3,2026-08-21,126
4,2026-08-28,334


In [6]:
normal = fridays["lines"].iloc[:4].mean()
today = fridays["lines"].iloc[4]
print(f"A normal Friday at S02: {normal:.0f} lines (the average of the four Fridays before).")
print(f"Friday 28 August:       {today} lines. That is {100 * today / normal:.0f}% of normal.")

A normal Friday at S02: 148 lines (the average of the four Fridays before).
Friday 28 August:       334 lines. That is 226% of normal.


A normal Friday at `S02` has about 148 lines. This one has 334: more than twice as many. (You know why from notebook 2: the file arrived twice.)

The number you compare with is called the **baseline**. It is what normal looks like. Here: the average of the same weekday in the four weeks before.

## Step 4: A normal day for every store and every day

Do the same for all stores and all days at once. Each store-day is joined to the same store 7, 14, 21 and 28 days earlier.

In [7]:
NORMAL_DAY = """
    SELECT today.store_id, today.sales_date, today.lines,
           count(before.lines) AS days_compared,
           round(avg(before.lines)) AS normal_lines,
           round(100.0 * today.lines / avg(before.lines)) AS percent_of_normal
    FROM (
        SELECT store_id, sales_date, count(*) AS lines FROM clean.sales_lines
        WHERE sales_date <= DATE '2026-08-30'
        GROUP BY store_id, sales_date
    ) today
    JOIN (
        SELECT store_id, sales_date, count(*) AS lines FROM clean.sales_lines
        GROUP BY store_id, sales_date
    ) before
      ON before.store_id = today.store_id
     AND before.sales_date IN (today.sales_date - 7, today.sales_date - 14,
                               today.sales_date - 21, today.sales_date - 28)
    GROUP BY today.store_id, today.sales_date, today.lines
"""
week = sql(f"SELECT * FROM ({NORMAL_DAY}) n WHERE sales_date >= DATE '2026-08-24' ORDER BY percent_of_normal")
print("The lowest six store-days of the bonus week:")
display(week.head(6))
print("The highest four:")
week.tail(4)

The lowest six store-days of the bonus week:


,store_id,sales_date,lines,days_compared,normal_lines,percent_of_normal
0,S02,2026-08-26,36,4,125,29
1,S06,2026-08-26,32,4,92,35
2,S01,2026-08-26,52,4,139,37
3,S05,2026-08-26,30,4,78,38
4,S03,2026-08-26,42,4,110,38
5,S04,2026-08-26,41,4,106,39


The highest four:


,store_id,sales_date,lines,days_compared,normal_lines,percent_of_normal
36,S04,2026-08-27,124,4,109,114
37,S03,2026-08-27,121,4,105,115
38,S02,2026-08-30,149,4,121,123
39,S02,2026-08-28,334,4,148,226


Six store-days sit near 30% to 40% of normal. Most of the rest are between 79% and 123%. One is at 226%.

## Step 5: Where is the line?

A check needs a line between "fine" and "look at this". Do not guess it. **Take it from days you know were good.**

The four weeks before the bonus week are clean. How far from normal does a good day get? (A day needs at least three earlier days to compare with, so only the last clean week can be measured.)

In [8]:
sql(f"""
    SELECT count(*) AS good_store_days,
           min(percent_of_normal) AS lowest_percent,
           max(percent_of_normal) AS highest_percent
    FROM ({NORMAL_DAY}) n
    WHERE days_compared >= 3 AND sales_date < DATE '2026-08-24'
""")

,good_store_days,lowest_percent,highest_percent
0,42,74,131


On 42 good store-days, the busiest was 131% of normal and the quietest 74%. So an ordinary day wanders about a quarter up or down.

Lotus puts the line a little outside that: **between 60% and 140% of the normal day**. Close enough to catch half a file or a doubled file. Far enough that an ordinary busy Saturday does not ring.

The line between good and bad in a check is called its **limit**. You will also hear **threshold**: it is the same thing.

Apply it.

In [9]:
alarms = sql(f"""
    SELECT store_id, sales_date, lines, normal_lines, percent_of_normal
    FROM ({NORMAL_DAY}) n
    WHERE days_compared >= 3 AND (percent_of_normal < 60 OR percent_of_normal > 140)
    ORDER BY sales_date, store_id
""")
alarms

,store_id,sales_date,lines,normal_lines,percent_of_normal
0,S01,2026-08-26,52,139,37
1,S02,2026-08-26,36,125,29
2,S03,2026-08-26,42,110,38
3,S04,2026-08-26,41,106,39
4,S05,2026-08-26,30,78,38
5,S06,2026-08-26,32,92,35
6,S02,2026-08-28,334,148,226


Seven rows. Each one is an **anomaly**: a value far from the baseline.

Be careful with that word. An anomaly is a reason to look. It is not a proof of a fault. The last row is a real fault: Dana's doubled Friday. Look at the other six. They are all the same day.

And notice what is **not** in the list: Ines's Tuesday. A store with no rows has nothing to compare. The volume check cannot see it. Only "every store reported", from step 2, can.

| Check | A doubled file | Half a file | No file at all |
|---|---|---|---|
| Near a normal day | Yes | Yes | **No**: nothing to compare |
| Every store reported | No | No | **Yes** |

You need both.

## What you now know

- A check on rows cannot see rows that are not there. Write down what you expect, and take away what you have.
- A baseline is the number you compare with. It is what normal looks like.
- Compare a store with **itself**, on the same weekday.
- A limit is the line between good and bad. Take it from days you know were good. It is also called a threshold.
- An anomaly is a value far from the baseline. It is a reason to look, not a proof.

**Say it in an interview:** "A store sent no file one night, and every check passed, because a missing row cannot fail a check. I added two operational checks: every open store has rows for the day, and each store's line count is within 60% to 140% of its own normal day, the average of the same weekday over four weeks. I took that band from clean history, where good days stayed between 74% and 131%."

**The theory:** [Data Observability & Monitoring](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-observability-and-monitoring), section "What is normal?". [Data Quality & Validation](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-quality-and-validation), section "Freshness and volume".

Next: [notebook 7](07_the_false_alarm.ipynb). You have nine alarms. Seven of them are wrong.